## Deploy a Model to Microsoft Foundry with Python

In this notebook, you will learn how to **deploy an AI model** to your Foundry project — entirely with Python code. Instead of clicking through the portal, we will:

1. Connect to Azure and list available models
2. Pick a model and deploy it programmatically
3. List all active deployments in our project
4. Test the deployed model by chatting with it

By the end, you will have a fully functional model endpoint that you can use from any of the other demos in this course!


### Step 1: Install the Python Packages We Need

We need two types of SDKs for this demo:
- **`azure-mgmt-cognitiveservices`** — the Azure management SDK that lets us create model deployments on our Cognitive Services resource
- **`azure-ai-projects`** — the Foundry SDK we have been using in other demos, which lets us list deployments and test them

We also install `azure-identity` (for secure login), `python-dotenv` (to load settings from a file), and `openai` (to test the deployed model).

In [ ]:
# Install all the packages we need:
# - azure-mgmt-cognitiveservices: manage deployments on Azure Cognitive Services
# - azure-ai-projects: list deployments and get an OpenAI client for testing
# - openai: the OpenAI library used behind the scenes for chat
# - python-dotenv: reads our .env configuration file
# - azure-identity: handles Azure login securely
%pip install azure-mgmt-cognitiveservices azure-ai-projects==2.0.0b2 openai==1.109.1 python-dotenv azure-identity

### Step 2: Load Our Configuration

Just like the other demos, we store sensitive information in a `.env` file. For this notebook we need a few extra values because the management SDK needs to know *which* Azure resource to talk to.

Your `.env` file should contain:
- `FOUNDRY_PROJECT_ENDPOINT` — the Foundry project URL (used later to test the model)
- `AZURE_SUBSCRIPTION_ID` — your Azure subscription ID
- `AZURE_RESOURCE_GROUP` — the resource group that contains your Foundry resource
- `FOUNDRY_ACCOUNT_NAME` — the name of your Azure AI Services / Cognitive Services account (the parent resource, not the project)

In [ ]:
# 'os' lets us access environment variables
import os

# 'load_dotenv' reads key-value pairs from a .env file into environment variables
from dotenv import load_dotenv

# Load the .env file — this must run before we read any values: NOTE: I use an absolute path here to avoid confusion about where the .env file should be located.
load_dotenv("/Users/macbookpro/dev/python/github/python/foundry-nett/00-Setup/.env")

# Read each setting from the environment
my_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")       # Foundry project URL (for testing later)
subscription_id = os.getenv("AZURE_SUBSCRIPTION_ID")       # Azure subscription ID
resource_group = os.getenv("AZURE_RESOURCE_GROUP")         # Resource group name
account_name = os.getenv("FOUNDRY_ACCOUNT_NAME")           # Cognitive Services account name

# Quick check — make sure nothing is missing
print("Endpoint:", my_endpoint)
print("Subscription:", subscription_id)
print("Resource group:", resource_group)
print("Account name:", account_name)

### Step 3: Connect to Azure

We need **two clients** in this notebook, each serving a different purpose:

| Client | SDK | What it does |
|---|---|---|
| `CognitiveServicesManagementClient` | `azure-mgmt-cognitiveservices` | List available models, create/delete deployments |
| `AIProjectClient` | `azure-ai-projects` | List existing deployments, test models via OpenAI client |

Think of the management client as the construction crew that *builds* the deployment, and `AIProjectClient` as the front desk that *uses* it.

In [ ]:
# DefaultAzureCredential handles Azure login automatically
# It tries multiple login methods (VS Code, Azure CLI, etc.) until one works
from azure.identity import DefaultAzureCredential

# CognitiveServicesManagementClient lets us manage deployments on our Azure AI resource
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient

# AIProjectClient is the Foundry SDK client — we use it to list and test deployments
from azure.ai.projects import AIProjectClient

# Create the credential object (shared by both clients)
credential = DefaultAzureCredential()

# --- Client 1: Management client (for deploying models) ---
# This client talks to the Azure Resource Manager (management plane)
mgmt_client = CognitiveServicesManagementClient(
    credential=credential,
    subscription_id=subscription_id,
)
print("Management client connected!")

# --- Client 2: AIProjectClient (for listing deployments and testing) ---
# This client talks to the Foundry project (data plane)
foundry_client = AIProjectClient(
    endpoint=my_endpoint,
    credential=credential,
)
print("AIProjectClient connected!")

### Step 4: Browse Available Models

Before deploying a model, let's see which ones are available in our Azure region. Not every model is available everywhere — this step shows you exactly what you can deploy.

We will filter for **OpenAI** format models (GPT, DALL-E, Whisper, embeddings, etc.) since those are the most commonly used.

In [ ]:
# List all models available for our Cognitive Services account
# This returns models that can be deployed in the region where our account lives
available_models = mgmt_client.accounts.list_models(
    resource_group_name=resource_group,
    account_name=account_name,
)

# Collect and display the models, grouped by format
# 'format' is usually 'OpenAI' for GPT models or other publisher names
print("Available models for deployment:\n")
model_count = 0
for model in available_models:
    model_count += 1
    # Show model name, version, and what it can do
    capabilities = ", ".join(model.capabilities.keys()) if model.capabilities else "N/A"
    print(f"  {model_count:3d}. {model.name:35s}  v{model.version:15s}  [{capabilities}]")
    # Stop after 30 to keep the output manageable
    if model_count >= 30:
        print(f"\n  ... showing first 30 of many models.")
        break

print(f"\nTip: Visit https://ai.azure.com/catalog to browse the full catalog with descriptions.")

### Step 5: Choose a Model to Deploy

Let's deploy **GPT-4.1 nano** — a fast, lightweight model from OpenAI that is great for simple tasks and costs very little to run.

> **Why GPT-4.1 nano?** It is quick to deploy, cheap to run, and perfect for demonstrating the deployment process. You can easily swap in a different model by changing the `model_name` and `model_version` below.

First, let's look up the exact details of this model to confirm it is available.

In [ ]:
# The model we want to deploy
# Change these values if you want to deploy a different model
model_name = "gpt-4.1-nano"           # The model name as it appears in the catalog
model_format = "OpenAI"                # The model format (OpenAI for GPT models)
model_version = "2025-04-14"           # The model version (check Step 4 output for available versions)

# The name we will give our deployment — this is how we reference it later
# Must be unique within our account and can contain letters, numbers, and hyphens
deployment_name = "gpt-41-nano-demo"

# Verify the model exists and find its supported SKU
# Different models support different SKUs (e.g., 'Standard', 'GlobalStandard', etc.)
# We will automatically pick the first available SKU so the deployment works correctly
found = False
model_sku_name = None
model_sku_capacity = None

for model in mgmt_client.accounts.list_models(
    resource_group_name=resource_group,
    account_name=account_name,
):
    if model.name == model_name and model.version == model_version:
        found = True
        capabilities = ", ".join(model.capabilities.keys()) if model.capabilities else "N/A"
        print(f"Model found!")
        print(f"  Name:         {model.name}")
        print(f"  Version:      {model.version}")
        print(f"  Format:       {model.format}")
        print(f"  Capabilities: {capabilities}")

        # Show all available SKUs and pick the first one for deployment
        if model.skus:
            sku_names = [s.name for s in model.skus]
            print(f"  Available SKUs: {', '.join(sku_names)}")

            # Use the first SKU — this is typically the recommended one
            first_sku = model.skus[0]
            model_sku_name = first_sku.name
            model_sku_capacity = first_sku.capacity.default if first_sku.capacity else 10
            print(f"\n  Selected SKU for deployment:")
            print(f"    SKU:      {model_sku_name}")
            print(f"    Capacity: {model_sku_capacity}")

        break

if not found:
    print(f"Model '{model_name}' version '{model_version}' was not found!")
    print("Check the output from Step 4 for available models and versions.")

### Step 6: Deploy the Model

This is the main event! We will create a deployment using the SKU that was automatically selected in Step 5. Different models support different SKUs — for example, `gpt-4.1-nano` uses **GlobalStandard** (a globally load-balanced pay-as-you-go tier), while some older models use **Standard**.

We need to specify:
- **Deployment name** — how we will reference it later
- **Model details** — name, version, and format
- **SKU** — the pricing tier (auto-detected from Step 5)
- **Capacity** — the rate limit in thousands of tokens per minute (TPM)

In [ ]:
# Import the classes we need to define the deployment
from azure.mgmt.cognitiveservices.models import (
    Deployment,              # The top-level deployment object
    DeploymentProperties,    # Configuration properties (model, version upgrade policy, etc.)
    DeploymentModel,         # Specifies which model to deploy
    Sku,                     # The pricing tier and capacity
)

# Build the deployment definition
# This is like filling out a form that tells Azure exactly what to create
deployment = Deployment(
    properties=DeploymentProperties(
        model=DeploymentModel(
            name=model_name,             # Which model (e.g., 'gpt-4.1-nano')
            version=model_version,       # Which version (e.g., '2025-04-14')
            format=model_format,         # Which format (e.g., 'OpenAI')
        ),
        # version_upgrade_option controls automatic updates:
        #   'NoAutoUpgrade' = stay on this exact version until we change it manually
        #   'OnceNewDefaultVersionAvailable' = auto-upgrade when a new default is released
        version_upgrade_option="NoAutoUpgrade",
    ),
    sku=Sku(
        name=model_sku_name,         # SKU auto-detected in Step 5 (e.g., 'GlobalStandard')
        capacity=model_sku_capacity,  # Default capacity from the model's SKU definition
    ),
)

print(f"Deployment configuration:")
print(f"  Model:    {model_name} v{model_version}")
print(f"  SKU:      {model_sku_name}")
print(f"  Capacity: {model_sku_capacity}")

# Submit the deployment request and wait for it to finish
# begin_create_or_update() starts the process; .result() waits until it is done
print(f"\nDeploying '{model_name}' as '{deployment_name}'...")
print("This usually takes about 1 minute. Please wait...\n")

result = mgmt_client.deployments.begin_create_or_update(
    resource_group_name=resource_group,
    account_name=account_name,
    deployment_name=deployment_name,
    deployment=deployment,
).result()

# Print the result
print(f"Deployment complete!")
print(f"  Name:   {result.name}")
print(f"  Model:  {result.properties.model.name} v{result.properties.model.version}")
print(f"  Status: {result.properties.provisioning_state}")
print(f"  SKU:    {result.sku.name} (capacity: {result.sku.capacity})")

### Step 7: List All Deployments in Your Project

Now let's use the `AIProjectClient` (the Foundry SDK) to see all the models currently deployed in your project — including the one we just created. This is the same SDK you have been using in the other demos.

In [ ]:
# List all model deployments in the Foundry project
# This shows everything — models deployed from the portal AND from code
print("All model deployments in your Foundry project:\n")

for dep in foundry_client.deployments.list():
    print(f"  Name:      {dep.name}")
    print(f"  Model:     {dep.model_name}")
    print(f"  Publisher: {dep.model_publisher}")
    print(f"  Version:   {dep.model_version}")
    print(f"  Type:      {dep.type}")
    print()

### Step 8: Test the Deployed Model

The moment of truth! Let's send a question to the model we just deployed and see if it responds. We use the same `AIProjectClient` → `get_openai_client()` pattern you learned in the *Chat with an OpenAI Model* demo.

In [ ]:
# Get an OpenAI-compatible client from our Foundry client
# This handles authentication automatically — no API keys needed
chat_client = foundry_client.get_openai_client()

# Send a test question to our newly deployed model
# We use the deployment_name as the 'model' parameter (not the catalog model name)
print(f"Sending a test question to '{deployment_name}'...\n")

ai_response = chat_client.responses.create(
    model=deployment_name,               # Use the deployment name we chose in Step 5
    instructions="You are a helpful and concise assistant.",
    input="Explain what a model deployment is in Foundry in two sentences."
)

# Print the model's response
print(f"The model replied:\n{ai_response.output_text}")